In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
file_path = '/content/drive/MyDrive/Res/final_processed.csv'
if os.path.exists(file_path):
    print(f"The file '{file_path}' exists.")
else:
    print(f"The file '{file_path}' does not exist.")

In [ ]:
import pandas as pd
import torch

# Verify GPU
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Change runtime before proceeding.")

# Verify dataset
df = pd.read_csv(file_path)
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"\nSample row:")
print(df['content'].iloc[0][:200])

In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_token'))
print("Logged in to HuggingFace.")

In [ ]:
df = df[['content','label']].dropna()
df = df[df['content'].str.strip().str.len() > 10]
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_df, val_df = train_test_split(
    train_df, test_size=0.1, random_state=42,
    stratify=train_df['label']
)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
  return tokenizer(
        batch['content'],
        truncation=True,
        max_length=512,
        padding='max_length'
    )
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df.reset_index(drop=True))
train_dataset = train_dataset.map(tokenize, batched=True, batch_size=64)
val_dataset   = val_dataset.map(tokenize, batched=True, batch_size=64)
test_dataset  = test_dataset.map(tokenize, batched=True, batch_size=64)

cols = ['input_ids', 'attention_mask', 'label']
train_dataset.set_format(type='torch', columns=cols)
val_dataset.set_format(type='torch', columns=cols)
test_dataset.set_format(type='torch', columns=cols)

print("Tokenization complete.")
print(f"Sample shape: {train_dataset[0]['input_ids'].shape}")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Real", 1: "Fake"},
    label2id={"Real": 0, "Fake": 1}
)
print("Model loaded.")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted')
    }

In [ ]:
training_args = TrainingArguments(
    output_dir='distilbert_fakenews',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=100,
    fp16=torch.cuda.is_available()
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# Cell — Final evaluation on test set
results = trainer.evaluate(test_dataset)
print(f"Test Accuracy: {results['eval_accuracy']:.4f}")
print(f"Test F1:       {results['eval_f1']:.4f}")

In [ ]:
# Cell — Save model
trainer.save_model('distilbert_finetuned')
tokenizer.save_pretrained('distilbert_finetuned')
print("Saved.")

In [ ]:
# Cell — Download
!zip -r distilbert_finetuned.zip distilbert_finetuned/
from google.colab import files
files.download('distilbert_finetuned.zip')

In [ ]:
import nbformat
import json
notebook_path = '/content/drive/MyDrive/Colab_Notebooks/Bert_model.ipynb'
# Load the notebook
with open(notebook_path, 'r') as f:
    nb = json.load(f)

# Fix the metadata.widgets issue
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

# Clear all outputs to reduce file size too
for cell in nb['cells']:
    if cell['cell_type'] == 'code':
        cell['outputs'] = []
        cell['execution_count'] = None

# Save fixed notebook
with open(notebook_path, 'w') as f:
    json.dump(nb, f, indent=1)

print("Fixed.")